# ベルマン方程式（期待方程式・最適方程式）

ベルマン方程式（期待方程式・最適方程式）では、更新式が長期報酬の見積もりにどう効くかを、数式とコードの両方で確認します。


## 参考動画
- [対応動画 1](https://www.youtube.com/watch?v=7nRgyYjMpas)
@[youtube](https://www.youtube.com/watch?v=7nRgyYjMpas)
- [対応動画 2](https://www.youtube.com/watch?v=BWcXCecwLcI)

## 参考リンク
- https://www.youtube.com/watch?v=7nRgyYjMpas
- https://www.youtube.com/watch?v=BWcXCecwLcI


## 背景と目的

ベルマン方程式は、長期最適化を局所更新に分解するための基礎式です。

期待方程式と最適方程式の違いを理解すると、動的計画法と学習法を接続できます。

再帰式を反復計算して収束の意味を確認します。


## 最初に解きたい疑問

1. 期待方程式と最適方程式は、どこが数学的に違うのか。
2. `max` が入ると、何が「最適」になるのか。
3. なぜ局所的な更新だけで、長期報酬を表せるのか。
4. Q学習の更新式は、ベルマン最適方程式とどうつながるのか。
5. 探索率の話が、この章で出てくる理由は何なのか。


## 先に押さえる言葉

- `Bellman expectation equation`: ある方策に従ったときの価値を、現在の報酬と次状態の価値で表す式です。方策を固定して評価するときに使います。
- `Bellman optimality equation`: 最もよい行動を選ぶ前提で、最適価値を再帰的に表した式です。`max` が入るのが特徴です。
- `bootstrap`: 現在の推定値を、次の推定値の材料として使う考え方です。すべての未来を待たずに学習を進められます。
- `greedy action`: 今見えている価値が最も高い行動を選ぶことです。探索を混ぜない純粋な選択です。


## 実行前の見取り図

1. `mc_return=` が、終端まで足し切った参照値として出ているか。
2. `updated V(s)=` が、1回のベルマン更新の結果になっているか。
3. `Q(s0,right)=` と `choose_action(...)` の出力で、更新と行動選択の両方が見えているか。


## 出力の読み方

- 表示されているのは full return で、ブートストラップした目標値ではない。


## つまずきやすい点

- 期待方程式、最適方程式、Q更新の関係を、1つの図か表で整理する説明がない。
- `p(s',r\mid s,a)` の確率和が、コードではどう簡略化されているかの説明が足りない。
- 終端まで足し切った値と、bootstrapped target の違いがまだ曖昧。


## このノートの守備範囲

このノートでは次の点は入口だけ触れるか、別ノートに分けて扱います。

- このセルはベルマン更新の前提確認であって、bootstrapping の実演ではない。


## 観察 1: ベルマン更新前の準備

ベルマン方程式へ入る前に、割引率と報酬列を固定して再帰構造を確認します。


In [ ]:
rewards = [1.0, 0.4, 0.0, 1.2]
gamma = 0.92
g = 0.0
for r in reversed(rewards):
    g = r + gamma * g
print('task = bellman-equations', 'mc_return=', round(g, 6))


この `mc_return` は終端までの実報酬を足し切った Monte Carlo 参照値です。ベルマン更新で使う bootstrapped target は、次状態価値 `V(s')` や `Q(s', a')` を入れた別の量です。


再帰的な更新規則を次セルで定量的に確認します。



## 観察 2: ベルマン更新を1回行う

次に、価値更新を1ステップだけ計算します。1回更新でも、再帰構造の意味は十分に見えてきます。


In [ ]:
v_next = {'s0': 0.4, 's1': 0.8}
reward = {'left': 0.2, 'right': 1.0}
trans = {'left': 's0', 'right': 's1'}
v_s = max(reward[a] + gamma * v_next[trans[a]] for a in ['left', 'right'])
print('updated V(s)=', round(v_s, 6))

ベルマン更新は『今の価値』を『次状態の価値』で再定義する操作です。この再帰が強化学習の中心です。



## 計算の対応表

1. $V^{\pi}(s) = \sum_a \pi(a\mid s)\sum_{s',r} p(s',r\mid s,a)\,[r + \gamma V^{\pi}(s')]$
2. $V^*(s) = \max_a\sum_{s',r} p(s',r\mid s,a)\,[r + \gamma V^*(s')]$


## 観察 3: Q値更新を比較する

ここで Q学習の更新式をコードに写し、数値の動きを確認します。式を読むだけでは掴みにくい感覚を得る段階です。


In [ ]:
Q = {('s0','left'): 0.3, ('s0','right'): 0.1, ('s1','left'): 0.5, ('s1','right'): 0.7}
alpha = 0.2
r, s, a, s_next = 1.0, 's0', 'right', 's1'
td_target = r + gamma * max(Q[(s_next,'left')], Q[(s_next,'right')])
Q[(s,a)] += alpha * (td_target - Q[(s,a)])
print('Q(s0,right)=', round(Q[(s,a)], 6))

更新後の値が過去の値とどれだけ違うかは、学習率と TD 誤差で決まります。ここが調整ポイントです。



## 観察 4: 探索と活用の切り替え

次に、探索率を変えたときの行動選択を見ます。探索不足は局所最適に閉じる典型的な原因です。


In [ ]:
import random

random.seed(13)


def choose_action(q_left, q_right, epsilon):
    greedy = 'left' if q_left >= q_right else 'right'
    if random.random() < epsilon:
        action = random.choice(['left', 'right'])
        return {'mode': 'explore', 'action': action, 'greedy': greedy}
    return {'mode': 'greedy', 'action': greedy, 'greedy': greedy}


for eps in [0.5, 0.1]:
    out = choose_action(0.4, 0.7, eps)
    print(f"epsilon={eps:.1f}", 'mode=', out['mode'], 'action=', out['action'], 'greedy=', out['greedy'])


探索率は固定せず、学習段階に応じて減衰させるのが一般的です。初期は広く探索し、後半で活用へ寄せます。



## 観察 5: 方策評価の簡易チェック

最後に、方策の平均報酬を簡易的に比較します。アルゴリズムの評価は、更新式だけでなく結果の検証が不可欠です。


In [ ]:
episode_rewards = [1.2, 0.8, 1.5, 1.1, 1.4]
avg_reward = sum(episode_rewards) / len(episode_rewards)
variance = sum((r - avg_reward) ** 2 for r in episode_rewards) / len(episode_rewards)
print('avg =', round(avg_reward, 4))
print('var =', round(variance, 4))

平均だけでなく分散を見ると、方策の安定性も評価できます。実運用ではこの二軸が重要です。



## 要点整理

今回のノートで押さえておくべき誤解しやすい点を整理します。

1. 探索率が低すぎて行動が固定化する
2. 報酬設計が目的とずれている
3. 長期ロールアウトの不安定性を検証しない
